# 04. 시나리오: 격자, 흔들기, key rate DV01

시나리오는 "리스크 요인 키 → 이동량" 데이터다. 생성기가 규칙으로 만들고, 시나리오 시장이 적용하고, 실행기가 평가한다(원칙 7).

| 단계 | QuantTool 코드 | ORE 원본 |
|---|---|---|
| 격자 | `configs/scenario/grid.yaml`, `src/quanttool/scenario/grid.py` | `OREAnalytics/orea/scenario/scenariosimmarketparameters.hpp` |
| 리스크 요인 키 | `src/quanttool/core/ids.py` `RiskFactorKey` | `QuantExt/qle/termstructures/scenario.hpp` |
| 시나리오 시장 | `src/quanttool/scenario/scenario_market.py` | `OREAnalytics/orea/scenario/scenariosimmarket.hpp` |
| 생성기 | `src/quanttool/scenario/generators.py` | `sensitivityscenariogenerator.hpp`, `historicalscenariogenerator.hpp` |
| 실행기 | `src/quanttool/scenario/runner.py` | `OREAnalytics/orea/engine/valuationengine.hpp` |

> 데이터: `tests/fixtures/ust_par_sample.csv`는 재무부 CSV와 **형식만 같은 합성 샘플**이다. 흐름을 익히는 용도이고, 숫자로 시장을 해석하지 않는다.

In [ ]:
from pathlib import Path

# 노트북이 어디서 열리든 레포 루트(pyproject.toml이 있는 곳)를 찾는다
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAMPLE_CSV = ROOT / "tests" / "fixtures" / "ust_par_sample.csv"  # 합성 샘플 (실제 시세 아님)

from datetime import date

import pandas as pd
import QuantLib as ql

from quanttool.analytics.sensitivity import key_rate_dv01, parallel_dv01
from quanttool.config import load_config
from quanttool.derivatives import Portfolio
from quanttool.market import Snapshot, UstParCsvLoader
from quanttool.scenario import (
    Scenario,
    ScenarioGrid,
    ScenarioMarket,
    ScenarioRunner,
    SensitivityGenerator,
)

cfg = load_config(ROOT / "configs")
snap = Snapshot.build(date(2026, 10, 23), cfg, UstParCsvLoader(SAMPLE_CSV))
book = Portfolio.load(ROOT / "positions" / "ust_book.yaml")

## 1. 격자: 흔들 수 있는 좌표
스냅샷 커브를 정해진 만기에서 제로금리로 샘플링한다. 각 점이 `RiskFactorKey` 하나다. PCA 로딩이나 과거 변화도 이 좌표로 정의한다(원칙 7-1).

In [ ]:
grid = ScenarioGrid(cfg.grid)
sample = grid.sample(snap)
pd.Series({str(k): v * 100 for k, v in sample.items()}, name="zero_cc_%").round(4)

## 2. 시나리오 시장: 격자 Quote로 다시 만든 커브
`ScenarioMarket`은 격자점마다 `SimpleQuote`를 두고, 그 사이를 제로금리 선형으로 이은 커브를 내준다. 원래 헬퍼·부트스트랩은 쓰지 않는다. 그래서 기준 커브와 격자점에서는 같고, 격자점 사이에서는 조금 다르다.

In [ ]:
smkt = ScenarioMarket(snap, grid)
with smkt.active():
    base_c, grid_c = snap.yield_curve("UST"), smkt.yield_curve("UST")
    rows = []
    for y in [0.5, 1.0, 1.25, 1.5, 2.0, 2.5, 5.0, 11.0, 27.0]:
        rows.append(
            {
                "years": y,
                "base_%": base_c.zeroRate(y, ql.Continuous).rate() * 100,
                "grid_%": grid_c.zeroRate(y, ql.Continuous).rate() * 100,
            }
        )
t = pd.DataFrame(rows).set_index("years")
t["diff_bp"] = (t["grid_%"] - t["base_%"]) * 100
t.round(4)

그 차이가 가격에 미치는 영향을 거래별로 대사한다 (`base_reconciliation`).

In [ ]:
smkt.base_reconciliation(book, cfg)

## 3. 시나리오를 손으로 만들어 적용하기
시나리오는 그냥 딕셔너리다. `apply`는 격자 Quote 값을 "기준 + 이동량"으로 바꾸고, `reset`은 되돌린다. 포트폴리오는 이미 시나리오 시장으로 조립되어 있다(위 대사 단계).

In [ ]:
keys = grid.keys("UST")
parallel = Scenario("parallel +10bp", {k: 0.0010 for k in keys})
with smkt.active():
    base = book.npvs()
    smkt.apply(parallel)
    shocked = book.npvs()
    smkt.reset()
(shocked - base).round(2)

### 예시: 스티프닝 (PCA 요인 시나리오의 모양)
PCA 시나리오는 "요인 크기 × 격자점별 로딩"이다. 아래 로딩은 **설명용으로 만든 값**(단기 −5bp → 장기 +5bp 선형)이고 추정값이 아니다. 실제 로딩은 `research/`에서 추정해 `configs/estimated/`에 출처와 함께 둔다.

In [ ]:
years = [
    ql.Actual365Fixed().yearFraction(ql.Date(23, 10, 2026), d)
    for d in grid.dates("UST", ql.Date(23, 10, 2026))
]
loading = {k: -1 + 2 * y / years[-1] for k, y in zip(keys, years, strict=True)}  # −1 … +1
steepener = Scenario("steepener 5bp", {k: 0.0005 * w for k, w in loading.items()})
with smkt.active():
    smkt.apply(steepener)
    pnl = book.npvs() - base
    smkt.reset()
pnl.round(2)

## 4. 생성기 + 실행기: key rate DV01
`SensitivityGenerator`가 격자점마다 +1bp 시나리오와 평행 +1bp 시나리오를 만든다. `ScenarioRunner`는 포트폴리오를 **한 번** 조립하고 시나리오마다 apply → NPV → reset을 반복한다.

In [ ]:
gen = SensitivityGenerator(grid, cfg.sensitivity)
scenarios = list(gen.generate())
print(
    len(scenarios),
    "개 시나리오, 예:",
    scenarios[7].label,
    {str(k): v for k, v in scenarios[7].shifts.items()},
)
res = ScenarioRunner(book, smkt, cfg).run(scenarios)

In [ ]:
krd = key_rate_dv01(res, cfg.sensitivity, "UST")
par = parallel_dv01(res, cfg.sensitivity, "UST")
print("Σ key rate DV01", round(krd["total"].sum(), 2), " / 평행 DV01", round(par["total"], 2))
krd.round(1)

## 정리와 다음 단계
- 시나리오 = 격자 좌표 위의 이동량. 규칙(민감도·스트레스·PCA·과거 변화)만 다르고 적용 방식은 같다.
- 지금 DV01은 **격자 제로금리 기준**이다. 헷지 수량(국채 par 기준)으로 바꾸는 것이 `scenario/par.py`(다음 단계).
- 과거 변화를 오늘에 적용하는 historical 시나리오는 날짜별 격자값 이력(`market/persist.py`)이 생기면 붙인다.